In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import torch.nn as nn
import pandas as pd

from utils.reproducibility_utils import seed_everything
from utils.data_loading import load_data, load_test_data

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# ============================================================================
# RNN VARIANTS (No GRU/LSTM)
# ============================================================================

class RNN(nn.Module):
    """Original Vanilla RNN (baseline)"""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class BiRNN(nn.Module):
    """Bidirectional RNN - sees future context"""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden, 1, nonlinearity='relu', 
                          batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden * 2, hidden)  # *2 for bidirectional
        self.out = nn.Linear(hidden, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class DeepRNN(nn.Module):
    """2-layer RNN - deeper model"""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden, num_layers=2, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class WideRNN(nn.Module):
    """RNN with 128 hidden units - wider model"""
    def __init__(self, input_size, hidden=128):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class WeightedL1(nn.Module):
    """Proven optimal: slope=1.0, threshold=2.0"""
    def forward(self, pred, target):
        weights = 1.0 + 1.0 * torch.clamp(torch.abs(target) - 2.0, min=0)
        return (torch.abs(pred - target) * weights).mean()


def predict(model, loader):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(x.to(device)).cpu().numpy() for x, _ in loader]).flatten()

print("✅ All RNN variants defined: RNN, BiRNN, DeepRNN, WideRNN")

In [ ]:
# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================

def train_two_stage(model_class, input_size, train_loader, val_loader, seed, 
                    s1_epochs=30, s2_epochs=40, s2_lr=0.00005):
    """Two-stage training with any model architecture"""
    seed_everything(seed)
    model = model_class(input_size).to(device)
    
    # Stage 1: L1 Loss
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    for _ in range(s1_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                  for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    s1_best = best
    
    # Stage 2: Weighted L1
    opt = torch.optim.Adam(model.parameters(), lr=s2_lr)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    for _ in range(s2_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                  for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    
    return model, s1_best, best2


def train_baseline(model_class, input_size, train_loader, val_loader, seed):
    """Standard L1 training"""
    seed_everything(seed)
    model = model_class(input_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    for _ in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                  for x,y in val_loader) / len(val_loader)
        if val < best: best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    model.load_state_dict(state)
    return model, best


def train_full_data(model_class, input_size, train_loader, seed, epochs=70):
    """Train on 100% data (no validation)"""
    seed_everything(seed)
    model = model_class(input_size).to(device)
    
    # Stage 1
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    for _ in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
    
    # Stage 2
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    crit2 = WeightedL1()
    for _ in range(40):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
    
    return model

print("✅ Training functions ready")

In [ ]:
# ============================================================================
# DATA LOADING FUNCTIONS
# ============================================================================

SEEDS = [42, 2024]

def load_standard_data(seq_len=20, include_vycg=False, train_ratio=0.98):
    """Load data with configurable options"""
    cols_to_drop = ["sideSlip", "time", "longitude", "latitude"]
    if not include_vycg:
        cols_to_drop.append("vyCG")
    
    train_loader, val_loader, pipeline = load_data(
        train_path="../datasets_challenge/train",
        train_ratio=train_ratio,
        batchsize=64,
        columns_to_drop=cols_to_drop,
        sequence_length=seq_len,
        threshold=True,
        threshold_value=3.0,
        threshold_column="vxCG",
        target="sideSlip",
        scaling=True,
        seed=42,
    )
    
    test_loader = load_test_data(
        test_path="../datasets_challenge/test",
        batchsize=512,
        pipeline=pipeline,
        load_anonymized_data=True
    )
    
    X_sample, _ = next(iter(train_loader))
    input_size = X_sample.shape[2]
    
    return train_loader, val_loader, test_loader, input_size, pipeline


def load_full_train_data(seq_len=20, include_vycg=False):
    """Load 100% training data (no validation split)"""
    cols_to_drop = ["sideSlip", "time", "longitude", "latitude"]
    if not include_vycg:
        cols_to_drop.append("vyCG")
    
    train_loader, _, pipeline = load_data(
        train_path="../datasets_challenge/train",
        train_ratio=1.0,  # 100% training!
        batchsize=64,
        columns_to_drop=cols_to_drop,
        sequence_length=seq_len,
        threshold=True,
        threshold_value=3.0,
        threshold_column="vxCG",
        target="sideSlip",
        scaling=True,
        seed=42,
    )
    
    test_loader = load_test_data(
        test_path="../datasets_challenge/test",
        batchsize=512,
        pipeline=pipeline,
        load_anonymized_data=True
    )
    
    X_sample, _ = next(iter(train_loader))
    input_size = X_sample.shape[2]
    
    return train_loader, test_loader, input_size

print("✅ Data loading functions ready")

## 🚀 Experiment 1: Different Sequence Lengths (3 files)

In [ ]:
print("="*60)
print("🚀 EXPERIMENT 1: DIFFERENT SEQUENCE LENGTHS")
print("="*60)

seq_results = {}

for seq_len in [15, 25, 30]:
    print(f"\n📏 Sequence length: {seq_len}")
    
    # Load data with different sequence length
    tr_loader, vl_loader, te_loader, inp_size, _ = load_standard_data(seq_len=seq_len)
    print(f"   Input size: {inp_size}")
    
    base_preds = []
    ts_preds = []
    
    for seed in SEEDS:
        model, _ = train_baseline(RNN, inp_size, tr_loader, vl_loader, seed)
        base_preds.append(predict(model, te_loader))
        
        model, s1, s2 = train_two_stage(RNN, inp_size, tr_loader, vl_loader, seed)
        ts_preds.append(predict(model, te_loader))
        print(f"   Seed {seed}: S1={s1:.5f}, S2={s2:.5f}")
    
    base_ens = np.mean(base_preds, axis=0)
    ts_ens = np.mean(ts_preds, axis=0)
    blend = (base_ens + ts_ens) / 2
    seq_results[seq_len] = blend
    
    # Save
    pd.DataFrame({"id": range(len(blend)), "predicted_sideslip": blend}).to_csv(
        f"pred_v11_seqlen{seq_len}.csv", index=False)
    print(f"   ✅ pred_v11_seqlen{seq_len}.csv")

## 🚀 Experiment 2: Include vyCG Feature (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 2: INCLUDE vyCG FEATURE")
print("="*60)

# Load data WITH vyCG
tr_vycg, vl_vycg, te_vycg, inp_vycg, _ = load_standard_data(seq_len=20, include_vycg=True)
print(f"Input size with vyCG: {inp_vycg}")

vycg_base_preds = []
vycg_ts_preds = []

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    
    model, val = train_baseline(RNN, inp_vycg, tr_vycg, vl_vycg, seed)
    vycg_base_preds.append(predict(model, te_vycg))
    print(f"   Baseline val: {val:.5f}")
    
    model, s1, s2 = train_two_stage(RNN, inp_vycg, tr_vycg, vl_vycg, seed)
    vycg_ts_preds.append(predict(model, te_vycg))
    print(f"   Two-stage: S1={s1:.5f}, S2={s2:.5f}")

vycg_base_ens = np.mean(vycg_base_preds, axis=0)
vycg_ts_ens = np.mean(vycg_ts_preds, axis=0)
vycg_blend = (vycg_base_ens + vycg_ts_ens) / 2

# File 4: With vyCG
pd.DataFrame({"id": range(len(vycg_blend)), "predicted_sideslip": vycg_blend}).to_csv(
    "pred_v11_with_vycg.csv", index=False)
print("\n✅ pred_v11_with_vycg.csv")

# Also need standard RNN for ensembles later
print("\n📦 Training standard RNN for comparison...")
tr_std, vl_std, te_std, inp_std, _ = load_standard_data(seq_len=20)
rnn_base_preds = []
rnn_ts_preds = []
for seed in SEEDS:
    model, _ = train_baseline(RNN, inp_std, tr_std, vl_std, seed)
    rnn_base_preds.append(predict(model, te_std))
    model, _, _ = train_two_stage(RNN, inp_std, tr_std, vl_std, seed)
    rnn_ts_preds.append(predict(model, te_std))

rnn_base_ens = np.mean(rnn_base_preds, axis=0)
rnn_ts_ens = np.mean(rnn_ts_preds, axis=0)
rnn_blend = (rnn_base_ens + rnn_ts_ens) / 2

# File 5: Ensemble with/without vyCG
vycg_ensemble = (vycg_blend + rnn_blend) / 2
pd.DataFrame({"id": range(len(vycg_ensemble)), "predicted_sideslip": vycg_ensemble}).to_csv(
    "pred_v11_vycg_ensemble.csv", index=False)
print("✅ pred_v11_vycg_ensemble.csv")

## 🚀 Experiment 3: Bidirectional RNN (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 3: BIDIRECTIONAL RNN")
print("="*60)

birnn_base_preds = []
birnn_ts_preds = []

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    
    model, val = train_baseline(BiRNN, inp_std, tr_std, vl_std, seed)
    birnn_base_preds.append(predict(model, te_std))
    print(f"   Baseline val: {val:.5f}")
    
    model, s1, s2 = train_two_stage(BiRNN, inp_std, tr_std, vl_std, seed)
    birnn_ts_preds.append(predict(model, te_std))
    print(f"   Two-stage: S1={s1:.5f}, S2={s2:.5f}")

birnn_base_ens = np.mean(birnn_base_preds, axis=0)
birnn_ts_ens = np.mean(birnn_ts_preds, axis=0)
birnn_blend = (birnn_base_ens + birnn_ts_ens) / 2

# File 6: BiRNN blend
pd.DataFrame({"id": range(len(birnn_blend)), "predicted_sideslip": birnn_blend}).to_csv(
    "pred_v11_birnn.csv", index=False)
print("\n✅ pred_v11_birnn.csv")

# File 7: BiRNN + RNN ensemble
birnn_rnn_ens = (birnn_blend + rnn_blend) / 2
pd.DataFrame({"id": range(len(birnn_rnn_ens)), "predicted_sideslip": birnn_rnn_ens}).to_csv(
    "pred_v11_birnn_rnn_ensemble.csv", index=False)
print("✅ pred_v11_birnn_rnn_ensemble.csv")

## 🚀 Experiment 4: Deeper RNN - 2 Layers (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 4: DEEPER RNN (2 Layers)")
print("="*60)

deep_base_preds = []
deep_ts_preds = []

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    
    model, val = train_baseline(DeepRNN, inp_std, tr_std, vl_std, seed)
    deep_base_preds.append(predict(model, te_std))
    print(f"   Baseline val: {val:.5f}")
    
    model, s1, s2 = train_two_stage(DeepRNN, inp_std, tr_std, vl_std, seed)
    deep_ts_preds.append(predict(model, te_std))
    print(f"   Two-stage: S1={s1:.5f}, S2={s2:.5f}")

deep_base_ens = np.mean(deep_base_preds, axis=0)
deep_ts_ens = np.mean(deep_ts_preds, axis=0)
deep_blend = (deep_base_ens + deep_ts_ens) / 2

# File 8: Deep RNN blend
pd.DataFrame({"id": range(len(deep_blend)), "predicted_sideslip": deep_blend}).to_csv(
    "pred_v11_deep_rnn.csv", index=False)
print("\n✅ pred_v11_deep_rnn.csv")

# File 9: Deep + Standard RNN ensemble
deep_std_ens = (deep_blend + rnn_blend) / 2
pd.DataFrame({"id": range(len(deep_std_ens)), "predicted_sideslip": deep_std_ens}).to_csv(
    "pred_v11_deep_std_ensemble.csv", index=False)
print("✅ pred_v11_deep_std_ensemble.csv")

## 🚀 Experiment 5: Wider RNN - 128 Hidden Units (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 5: WIDER RNN (128 Hidden Units)")
print("="*60)

wide_base_preds = []
wide_ts_preds = []

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    
    model, val = train_baseline(WideRNN, inp_std, tr_std, vl_std, seed)
    wide_base_preds.append(predict(model, te_std))
    print(f"   Baseline val: {val:.5f}")
    
    model, s1, s2 = train_two_stage(WideRNN, inp_std, tr_std, vl_std, seed)
    wide_ts_preds.append(predict(model, te_std))
    print(f"   Two-stage: S1={s1:.5f}, S2={s2:.5f}")

wide_base_ens = np.mean(wide_base_preds, axis=0)
wide_ts_ens = np.mean(wide_ts_preds, axis=0)
wide_blend = (wide_base_ens + wide_ts_ens) / 2

# File 10: Wide RNN blend
pd.DataFrame({"id": range(len(wide_blend)), "predicted_sideslip": wide_blend}).to_csv(
    "pred_v11_wide_rnn.csv", index=False)
print("\n✅ pred_v11_wide_rnn.csv")

# File 11: Wide + Standard RNN ensemble
wide_std_ens = (wide_blend + rnn_blend) / 2
pd.DataFrame({"id": range(len(wide_std_ens)), "predicted_sideslip": wide_std_ens}).to_csv(
    "pred_v11_wide_std_ensemble.csv", index=False)
print("✅ pred_v11_wide_std_ensemble.csv")

## 🚀 Experiment 6: 100% Training Data (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 6: 100% TRAINING DATA (No Validation)")
print("="*60)

# Load full training data
full_train, full_test, full_inp = load_full_train_data(seq_len=20)
print(f"Input size: {full_inp}")

full_preds = []
for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    model = train_full_data(RNN, full_inp, full_train, seed)
    full_preds.append(predict(model, full_test))
    print(f"   Training complete (no val - 100% data)")

full_ens = np.mean(full_preds, axis=0)

# File 12: 100% training
pd.DataFrame({"id": range(len(full_ens)), "predicted_sideslip": full_ens}).to_csv(
    "pred_v11_full_train.csv", index=False)
print("\n✅ pred_v11_full_train.csv")

# File 13: Blend full + standard
full_blend = (full_ens + rnn_blend) / 2
pd.DataFrame({"id": range(len(full_blend)), "predicted_sideslip": full_blend}).to_csv(
    "pred_v11_full_standard_blend.csv", index=False)
print("✅ pred_v11_full_standard_blend.csv")

## 🚀 Experiment 7: Different Train Ratios (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 7: DIFFERENT TRAIN RATIOS")
print("="*60)

ratio_results = {}

for ratio in [0.95, 0.99]:
    print(f"\n📊 Train ratio: {ratio}")
    
    tr_r, vl_r, te_r, inp_r, _ = load_standard_data(seq_len=20, train_ratio=ratio)
    
    base_preds = []
    ts_preds = []
    
    for seed in SEEDS:
        model, _ = train_baseline(RNN, inp_r, tr_r, vl_r, seed)
        base_preds.append(predict(model, te_r))
        
        model, s1, s2 = train_two_stage(RNN, inp_r, tr_r, vl_r, seed)
        ts_preds.append(predict(model, te_r))
        print(f"   Seed {seed}: S1={s1:.5f}, S2={s2:.5f}")
    
    base_ens = np.mean(base_preds, axis=0)
    ts_ens = np.mean(ts_preds, axis=0)
    blend = (base_ens + ts_ens) / 2
    ratio_results[ratio] = blend
    
    # Save
    ratio_str = str(ratio).replace(".", "")
    pd.DataFrame({"id": range(len(blend)), "predicted_sideslip": blend}).to_csv(
        f"pred_v11_ratio{ratio_str}.csv", index=False)
    print(f"   ✅ pred_v11_ratio{ratio_str}.csv")

In [ ]:
print("\n" + "="*60)
print("📋 SUMMARY: 15 PREDICTION FILES")
print("="*60)

print("\n🏆 Current Best: pred_v7_lr5e5_40ep → 0.21751")

print("\n📁 Sequence Length (3 files):")
print("  1. pred_v11_seqlen15.csv  - Shorter sequences")
print("  2. pred_v11_seqlen25.csv  - Longer sequences")
print("  3. pred_v11_seqlen30.csv  - Even longer")

print("\n📁 Feature Changes (2 files):")
print("  4. pred_v11_with_vycg.csv    - Include vyCG feature")
print("  5. pred_v11_vycg_ensemble.csv - With/without vyCG ensemble")

print("\n📁 Bidirectional RNN (2 files):")
print("  6. pred_v11_birnn.csv            - BiRNN 50/50")
print("  7. pred_v11_birnn_rnn_ensemble.csv - BiRNN+RNN ensemble")

print("\n📁 Deeper RNN (2 files):")
print("  8. pred_v11_deep_rnn.csv         - 2-layer RNN")
print("  9. pred_v11_deep_std_ensemble.csv - Deep+Standard ensemble")

print("\n📁 Wider RNN (2 files):")
print("  10. pred_v11_wide_rnn.csv         - 128 hidden units")
print("  11. pred_v11_wide_std_ensemble.csv - Wide+Standard ensemble")

print("\n📁 Training Data Changes (2 files):")
print("  12. pred_v11_full_train.csv         - 100% training data")
print("  13. pred_v11_full_standard_blend.csv - Full+standard blend")

print("\n📁 Train Ratio Changes (2 files):")
print("  14. pred_v11_ratio095.csv - 95% train")
print("  15. pred_v11_ratio099.csv - 99% train")

print("\n💡 Priority order for submission:")
print("  🥇 pred_v11_seqlen15.csv - Shorter might reduce overfitting")
print("  🥈 pred_v11_with_vycg.csv - vyCG is related to sideslip!")
print("  🥉 pred_v11_ratio099.csv - More training data")
print("  4. pred_v11_birnn.csv - See future context")
print("  5. pred_v11_full_train.csv - Maximum training data")

In [ ]:
# ============================================================================
# ALL MODEL ARCHITECTURES
# ============================================================================

class RNN(nn.Module):
    """Original Vanilla RNN (baseline)"""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden, 1, nonlinearity='relu', batch_first=True)
        self.fc = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class GRU(nn.Module):
    """GRU - better gradient flow than RNN"""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden, 1, batch_first=True)
        self.fc = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.gru(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class LSTM(nn.Module):
    """LSTM - memory cell for long sequences"""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden, 1, batch_first=True)
        self.fc = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.lstm(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class BiRNN(nn.Module):
    """Bidirectional RNN - sees future context"""
    def __init__(self, input_size, hidden=64):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden, 1, nonlinearity='relu', 
                          batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden * 2, hidden)  # *2 for bidirectional
        self.out = nn.Linear(hidden, 1)
        self.act = nn.ReLU()
    def forward(self, x):
        x, _ = self.rnn(x)
        return self.out(self.act(self.fc(x[:, -1, :])))


class WeightedL1(nn.Module):
    """Proven optimal: slope=1.0, threshold=2.0"""
    def forward(self, pred, target):
        weights = 1.0 + 1.0 * torch.clamp(torch.abs(target) - 2.0, min=0)
        return (torch.abs(pred - target) * weights).mean()


def predict(model, loader):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(x.to(device)).cpu().numpy() for x, _ in loader]).flatten()

print("✅ All architectures defined: RNN, GRU, LSTM, BiRNN")

In [ ]:
# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================

def train_two_stage(model_class, input_size, train_loader, val_loader, seed, 
                    s1_epochs=30, s2_epochs=40, s2_lr=0.00005):
    """Two-stage training with any model architecture"""
    seed_everything(seed)
    model = model_class(input_size).to(device)
    
    # Stage 1: L1 Loss
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state = np.inf, None
    for _ in range(s1_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                  for x,y in val_loader) / len(val_loader)
        if val < best: best, state = val, model.state_dict().copy()
    model.load_state_dict(state)
    s1_best = best
    
    # Stage 2: Weighted L1
    opt = torch.optim.Adam(model.parameters(), lr=s2_lr)
    crit2 = WeightedL1()
    best2, state2 = np.inf, None
    for _ in range(s2_epochs):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(nn.L1Loss()(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                  for x,y in val_loader) / len(val_loader)
        if val < best2: best2, state2 = val, model.state_dict().copy()
    model.load_state_dict(state2)
    
    return model, s1_best, best2


def train_baseline(model_class, input_size, train_loader, val_loader, seed):
    """Standard L1 training"""
    seed_everything(seed)
    model = model_class(input_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    best, state, patience = np.inf, None, 0
    for _ in range(50):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
        model.eval()
        val = sum(crit(model(x.to(device)), y.to(device).reshape(-1,1)).item() 
                  for x,y in val_loader) / len(val_loader)
        if val < best: best, state, patience = val, model.state_dict().copy(), 0
        else:
            patience += 1
            if patience > 10: break
    model.load_state_dict(state)
    return model, best


def train_full_data(model_class, input_size, train_loader, seed, epochs=70):
    """Train on 100% data (no validation)"""
    seed_everything(seed)
    model = model_class(input_size).to(device)
    
    # Stage 1
    opt = torch.optim.Adam(model.parameters(), lr=0.0003)
    crit = nn.L1Loss()
    for _ in range(30):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
    
    # Stage 2
    opt = torch.optim.Adam(model.parameters(), lr=0.00005)
    crit2 = WeightedL1()
    for _ in range(40):
        model.train()
        for x, y in train_loader:
            opt.zero_grad()
            crit2(model(x.to(device)), y.to(device).reshape(-1,1)).backward()
            opt.step()
    
    return model

print("✅ Training functions ready")

In [ ]:
# ============================================================================
# STANDARD DATA LOADING (baseline config)
# ============================================================================

SEEDS = [42, 2024]

def load_standard_data(seq_len=20, include_vycg=False):
    """Load data with configurable options"""
    cols_to_drop = ["sideSlip", "time", "longitude", "latitude"]
    if not include_vycg:
        cols_to_drop.append("vyCG")
    
    train_loader, val_loader, pipeline = load_data(
        train_path="../datasets_challenge/train",
        train_ratio=0.98,
        batchsize=64,
        columns_to_drop=cols_to_drop,
        sequence_length=seq_len,
        threshold=True,
        threshold_value=3.0,
        threshold_column="vxCG",
        target="sideSlip",
        scaling=True,
        seed=42,
    )
    
    test_loader = load_test_data(
        test_path="../datasets_challenge/test",
        batchsize=512,
        pipeline=pipeline,
        load_anonymized_data=True
    )
    
    X_sample, _ = next(iter(train_loader))
    input_size = X_sample.shape[2]
    
    return train_loader, val_loader, test_loader, input_size, pipeline


def load_full_train_data(seq_len=20, include_vycg=False):
    """Load 100% training data (no validation split)"""
    cols_to_drop = ["sideSlip", "time", "longitude", "latitude"]
    if not include_vycg:
        cols_to_drop.append("vyCG")
    
    train_loader, _, pipeline = load_data(
        train_path="../datasets_challenge/train",
        train_ratio=1.0,  # 100% training!
        batchsize=64,
        columns_to_drop=cols_to_drop,
        sequence_length=seq_len,
        threshold=True,
        threshold_value=3.0,
        threshold_column="vxCG",
        target="sideSlip",
        scaling=True,
        seed=42,
    )
    
    test_loader = load_test_data(
        test_path="../datasets_challenge/test",
        batchsize=512,
        pipeline=pipeline,
        load_anonymized_data=True
    )
    
    X_sample, _ = next(iter(train_loader))
    input_size = X_sample.shape[2]
    
    return train_loader, test_loader, input_size

print("✅ Data loading functions ready")

## 🚀 Experiment 1: GRU Architecture (3 files)

In [ ]:
print("="*60)
print("🚀 EXPERIMENT 1: GRU ARCHITECTURE")
print("="*60)

# Load standard data
train_loader, val_loader, test_loader, input_size, _ = load_standard_data(seq_len=20)
print(f"Input size: {input_size}")

# Train GRU models
gru_baseline_preds = []
gru_twostage_preds = []

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    
    # Baseline
    model, val = train_baseline(GRU, input_size, train_loader, val_loader, seed)
    gru_baseline_preds.append(predict(model, test_loader))
    print(f"   Baseline val: {val:.5f}")
    
    # Two-stage
    model, s1, s2 = train_two_stage(GRU, input_size, train_loader, val_loader, seed)
    gru_twostage_preds.append(predict(model, test_loader))
    print(f"   Two-stage val: S1={s1:.5f}, S2={s2:.5f}")

# Create ensembles
gru_base_ens = np.mean(gru_baseline_preds, axis=0)
gru_ts_ens = np.mean(gru_twostage_preds, axis=0)

# Save predictions
# File 1: GRU two-stage only
pd.DataFrame({"id": range(len(gru_ts_ens)), "predicted_sideslip": gru_ts_ens}).to_csv(
    "pred_v11_gru_twostage.csv", index=False)
print("\n✅ pred_v11_gru_twostage.csv")

# File 2: GRU 50/50 blend
gru_blend = (gru_base_ens + gru_ts_ens) / 2
pd.DataFrame({"id": range(len(gru_blend)), "predicted_sideslip": gru_blend}).to_csv(
    "pred_v11_gru_blend5050.csv", index=False)
print("✅ pred_v11_gru_blend5050.csv")

# File 3: GRU + RNN ensemble (cross-architecture)
# First train RNN for comparison
rnn_baseline_preds = []
rnn_twostage_preds = []
for seed in SEEDS:
    model, _ = train_baseline(RNN, input_size, train_loader, val_loader, seed)
    rnn_baseline_preds.append(predict(model, test_loader))
    model, _, _ = train_two_stage(RNN, input_size, train_loader, val_loader, seed)
    rnn_twostage_preds.append(predict(model, test_loader))

rnn_base_ens = np.mean(rnn_baseline_preds, axis=0)
rnn_ts_ens = np.mean(rnn_twostage_preds, axis=0)
rnn_blend = (rnn_base_ens + rnn_ts_ens) / 2

# Cross-architecture ensemble
cross_arch = (gru_blend + rnn_blend) / 2
pd.DataFrame({"id": range(len(cross_arch)), "predicted_sideslip": cross_arch}).to_csv(
    "pred_v11_gru_rnn_ensemble.csv", index=False)
print("✅ pred_v11_gru_rnn_ensemble.csv")

## 🚀 Experiment 2: LSTM Architecture (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 2: LSTM ARCHITECTURE")
print("="*60)

lstm_baseline_preds = []
lstm_twostage_preds = []

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    
    model, val = train_baseline(LSTM, input_size, train_loader, val_loader, seed)
    lstm_baseline_preds.append(predict(model, test_loader))
    print(f"   Baseline val: {val:.5f}")
    
    model, s1, s2 = train_two_stage(LSTM, input_size, train_loader, val_loader, seed)
    lstm_twostage_preds.append(predict(model, test_loader))
    print(f"   Two-stage val: S1={s1:.5f}, S2={s2:.5f}")

lstm_base_ens = np.mean(lstm_baseline_preds, axis=0)
lstm_ts_ens = np.mean(lstm_twostage_preds, axis=0)
lstm_blend = (lstm_base_ens + lstm_ts_ens) / 2

# File 4: LSTM blend
pd.DataFrame({"id": range(len(lstm_blend)), "predicted_sideslip": lstm_blend}).to_csv(
    "pred_v11_lstm_blend5050.csv", index=False)
print("\n✅ pred_v11_lstm_blend5050.csv")

# File 5: LSTM + RNN + GRU ensemble
triple_arch = (lstm_blend + rnn_blend + gru_blend) / 3
pd.DataFrame({"id": range(len(triple_arch)), "predicted_sideslip": triple_arch}).to_csv(
    "pred_v11_triple_arch_ensemble.csv", index=False)
print("✅ pred_v11_triple_arch_ensemble.csv")

## 🚀 Experiment 3: Bidirectional RNN (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 3: BIDIRECTIONAL RNN")
print("="*60)

birnn_baseline_preds = []
birnn_twostage_preds = []

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    
    model, val = train_baseline(BiRNN, input_size, train_loader, val_loader, seed)
    birnn_baseline_preds.append(predict(model, test_loader))
    print(f"   Baseline val: {val:.5f}")
    
    model, s1, s2 = train_two_stage(BiRNN, input_size, train_loader, val_loader, seed)
    birnn_twostage_preds.append(predict(model, test_loader))
    print(f"   Two-stage val: S1={s1:.5f}, S2={s2:.5f}")

birnn_base_ens = np.mean(birnn_baseline_preds, axis=0)
birnn_ts_ens = np.mean(birnn_twostage_preds, axis=0)
birnn_blend = (birnn_base_ens + birnn_ts_ens) / 2

# File 6: BiRNN blend
pd.DataFrame({"id": range(len(birnn_blend)), "predicted_sideslip": birnn_blend}).to_csv(
    "pred_v11_birnn_blend5050.csv", index=False)
print("\n✅ pred_v11_birnn_blend5050.csv")

# File 7: All 4 architectures
quad_arch = (rnn_blend + gru_blend + lstm_blend + birnn_blend) / 4
pd.DataFrame({"id": range(len(quad_arch)), "predicted_sideslip": quad_arch}).to_csv(
    "pred_v11_quad_arch_ensemble.csv", index=False)
print("✅ pred_v11_quad_arch_ensemble.csv")

## 🚀 Experiment 4: Different Sequence Lengths (3 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 4: DIFFERENT SEQUENCE LENGTHS")
print("="*60)

seq_results = {}

for seq_len in [15, 25, 30]:
    print(f"\n📏 Sequence length: {seq_len}")
    
    # Load data with different sequence length
    tr_loader, vl_loader, te_loader, inp_size, _ = load_standard_data(seq_len=seq_len)
    print(f"   Input size: {inp_size}")
    
    base_preds = []
    ts_preds = []
    
    for seed in SEEDS:
        model, _ = train_baseline(RNN, inp_size, tr_loader, vl_loader, seed)
        base_preds.append(predict(model, te_loader))
        
        model, _, s2 = train_two_stage(RNN, inp_size, tr_loader, vl_loader, seed)
        ts_preds.append(predict(model, te_loader))
        print(f"   Seed {seed}: S2 val={s2:.5f}")
    
    base_ens = np.mean(base_preds, axis=0)
    ts_ens = np.mean(ts_preds, axis=0)
    blend = (base_ens + ts_ens) / 2
    seq_results[seq_len] = blend
    
    # Save
    pd.DataFrame({"id": range(len(blend)), "predicted_sideslip": blend}).to_csv(
        f"pred_v11_seqlen{seq_len}.csv", index=False)
    print(f"   ✅ pred_v11_seqlen{seq_len}.csv")

## 🚀 Experiment 5: Include vyCG Feature (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 5: INCLUDE vyCG FEATURE")
print("="*60)

# Load data WITH vyCG
tr_vycg, vl_vycg, te_vycg, inp_vycg, _ = load_standard_data(seq_len=20, include_vycg=True)
print(f"Input size with vyCG: {inp_vycg}")

vycg_base_preds = []
vycg_ts_preds = []

for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    
    model, val = train_baseline(RNN, inp_vycg, tr_vycg, vl_vycg, seed)
    vycg_base_preds.append(predict(model, te_vycg))
    print(f"   Baseline val: {val:.5f}")
    
    model, s1, s2 = train_two_stage(RNN, inp_vycg, tr_vycg, vl_vycg, seed)
    vycg_ts_preds.append(predict(model, te_vycg))
    print(f"   Two-stage: S1={s1:.5f}, S2={s2:.5f}")

vycg_base_ens = np.mean(vycg_base_preds, axis=0)
vycg_ts_ens = np.mean(vycg_ts_preds, axis=0)
vycg_blend = (vycg_base_ens + vycg_ts_ens) / 2

# File 11: With vyCG
pd.DataFrame({"id": range(len(vycg_blend)), "predicted_sideslip": vycg_blend}).to_csv(
    "pred_v11_with_vycg.csv", index=False)
print("\n✅ pred_v11_with_vycg.csv")

# File 12: Ensemble with/without vyCG
vycg_ensemble = (vycg_blend + rnn_blend) / 2
pd.DataFrame({"id": range(len(vycg_ensemble)), "predicted_sideslip": vycg_ensemble}).to_csv(
    "pred_v11_vycg_ensemble.csv", index=False)
print("✅ pred_v11_vycg_ensemble.csv")

## 🚀 Experiment 6: 100% Training Data (2 files)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 6: 100% TRAINING DATA (No Validation)")
print("="*60)

# Load full training data
full_train, full_test, full_inp = load_full_train_data(seq_len=20)
print(f"Input size: {full_inp}")

full_preds = []
for seed in SEEDS:
    print(f"\n🌱 Seed {seed}")
    model = train_full_data(RNN, full_inp, full_train, seed)
    full_preds.append(predict(model, full_test))
    print(f"   Training complete (no val loss - 100% data)")

full_ens = np.mean(full_preds, axis=0)

# File 13: 100% training
pd.DataFrame({"id": range(len(full_ens)), "predicted_sideslip": full_ens}).to_csv(
    "pred_v11_full_train.csv", index=False)
print("\n✅ pred_v11_full_train.csv")

# File 14: Blend full + standard
full_blend = (full_ens + rnn_blend) / 2
pd.DataFrame({"id": range(len(full_blend)), "predicted_sideslip": full_blend}).to_csv(
    "pred_v11_full_standard_blend.csv", index=False)
print("✅ pred_v11_full_standard_blend.csv")

## 🚀 Experiment 7: Best Combination (1 file)

In [ ]:
print("\n" + "="*60)
print("🚀 EXPERIMENT 7: MEGA ENSEMBLE")
print("="*60)

# Combine best from each experiment
# RNN (original best) + GRU + different seq lengths
mega_ensemble = (
    rnn_blend +      # Original architecture
    gru_blend +      # Best alternative architecture (likely)
    seq_results[15] + seq_results[25]  # Different sequence lengths
) / 4

# File 15: Mega ensemble
pd.DataFrame({"id": range(len(mega_ensemble)), "predicted_sideslip": mega_ensemble}).to_csv(
    "pred_v11_mega_ensemble.csv", index=False)
print("✅ pred_v11_mega_ensemble.csv")

In [ ]:
print("\n" + "="*60)
print("📋 SUMMARY: 15 PREDICTION FILES")
print("="*60)

print("\n🏆 Current Best: pred_v7_lr5e5_40ep → 0.21751")

print("\n📁 Architecture Changes (7 files):")
print("  1. pred_v11_gru_twostage.csv       - GRU only")
print("  2. pred_v11_gru_blend5050.csv      - GRU 50/50")
print("  3. pred_v11_gru_rnn_ensemble.csv   - GRU+RNN")
print("  4. pred_v11_lstm_blend5050.csv     - LSTM 50/50")
print("  5. pred_v11_triple_arch_ensemble.csv - RNN+GRU+LSTM")
print("  6. pred_v11_birnn_blend5050.csv    - BiRNN 50/50")
print("  7. pred_v11_quad_arch_ensemble.csv - All 4 architectures")

print("\n📁 Sequence Length Changes (3 files):")
print("  8. pred_v11_seqlen15.csv  - Shorter sequences")
print("  9. pred_v11_seqlen25.csv  - Longer sequences")
print("  10. pred_v11_seqlen30.csv - Even longer")

print("\n📁 Feature Changes (2 files):")
print("  11. pred_v11_with_vycg.csv    - Include vyCG feature")
print("  12. pred_v11_vycg_ensemble.csv - With/without vyCG ensemble")

print("\n📁 Training Data Changes (2 files):")
print("  13. pred_v11_full_train.csv         - 100% training data")
print("  14. pred_v11_full_standard_blend.csv - Full+standard blend")

print("\n📁 Mega Ensemble (1 file):")
print("  15. pred_v11_mega_ensemble.csv - Best of everything")

print("\n💡 Priority order for submission:")
print("  🥇 pred_v11_gru_blend5050.csv - GRU often beats RNN")
print("  🥈 pred_v11_gru_rnn_ensemble.csv - Diversity helps")
print("  🥉 pred_v11_seqlen15.csv - Shorter might be better")
print("  4. pred_v11_mega_ensemble.csv")
print("  5. pred_v11_with_vycg.csv")